#  **Data**

In [18]:
import csv
import os
import urllib.request
import math
import lang2vec.lang2vec as l2v
import sys
from pathlib import Path

sys.path.append(str(Path("../src").resolve()))

from lang_distance import (
    phoneme_inventory_distance, script_row, normalize,
    lang2lang_average, closest_languages,
)
from uriel_distance import uriel_row, covered_languages
from script_distance import script_row_auto

### **Phoible Data**

In [19]:
_PHOIBLE_FEATURES = {
    "InventoryID", "Glottocode", "ISO6393", "LanguageName", "SpecificDialect",
    "GlyphID", "Phoneme", "Allophones", "Marginal", "SegmentClass", "Source",
}

phoible_path = Path("../data/phoible.csv")

reader = csv.DictReader(open(phoible_path, encoding="utf-8"))
features = [c for c in reader.fieldnames if c not in _PHOIBLE_FEATURES]

by_lang = {}
id2name = {}
for row in reader:
    # Language ID
    lid = row["ISO6393"]
    id2name.setdefault(lid, row["LanguageName"].lower())

    if not lid or lid == "NA":
        continue

    vec = [
        (row[f].split(",")[0] if row else 0)
        for f in features
    ]

    by_lang.setdefault(lid, {}).setdefault(row["InventoryID"], []).append(vec)

phoible = {}

for lid, inventories in by_lang.items():
    phoible[lid] = max(inventories.values(), key=len)

## **Lang2Lang**

In [20]:
target = "tgl" # maybe FIL also later?
k =  100
candidates = None
script_samples = None

METRIC_ORDER = ["phoneme", "genetic", "geographic", "syntactic", "phonological", "script"]
SHORT = {"phoneme": "inv", "genetic": "gen", "geographic": "geo",
         "syntactic": "syn", "phonological": "phon", "script": "scr"}

In [21]:
URIEL_METRICS = ("genetic", "geographic", "syntactic", "phonological")

if target not in phoible:
    raise ValueError(f"{target!r} has no Phoible inventory")

pool = set(phoible) & covered_languages()
if candidates is not None:
    pool &= set(candidates)

pool.discard(target)

target_inv = phoible[target]
cand_inv = {c: phoible[c] for c in pool if c in phoible}

# URIEL

rows = {"phoneme": phoneme_inventory_distance(target_inv, cand_inv)}
for m in URIEL_METRICS:
    rows[m] = uriel_row(m, target, pool)

# SCRIPTS

# if script_samples and target in script_samples:
#     others = {c: script_samples[c] for c in pool if c in script_samples}
#     rows["scripts"] = script_row(script_samples[target], others)

rows["script"] = script_row_auto(target, pool)

base = set(rows["phoneme"])
for m in URIEL_METRICS:
    base &= set(rows[m])

norm = {}
for m, row in rows.items():
    sub = {c: row[c] for c in base if c in row}
    if sub:
        norm[m] = normalize(sub)

scores = lang2lang_average(norm.values())
close_langs = closest_languages(scores, k, base)
ranked = closest_languages(scores, k=k, candidates=base)
out = []
for lid, avg in ranked:
    per = {m: norm[m][lid] for m in norm if lid in norm[m]}
    out.append((lid, avg, per))


In [22]:
cols = [m for m in METRIC_ORDER]
header = f"{'lang':<6}" + "".join(f"{SHORT[m]:>7}" for m in cols) + f"{'AVG':>8}"
print(f"closest languages to {target!r}  (lower = more similar)")
print(header)
print("-" * len(header))
for lid, avg, per in out:
    cells = "".join(
        (f"{per[m]:>7.2f}" if m in per else f"{'-':>7}") for m in cols
    )
    print(f"{lid:<6}{cells}{avg:>8.3f}")

closest languages to 'tgl'  (lower = more similar)
lang      inv    gen    geo    syn   phon    scr     AVG
--------------------------------------------------------
tsg      0.00   0.13   0.01   0.07   0.00   0.00   0.034
cha      0.00   0.15   0.04   0.04   0.00   0.00   0.039
ivv      0.00   0.27   0.00   0.07   0.00   0.00   0.056
tiy      0.00   0.27   0.00   0.07   0.00   0.00   0.056
ceb      0.00   0.00   0.00   0.04   0.32   0.00   0.060
mrw      0.22   0.12   0.00   0.02   0.08   0.00   0.075
akl      0.00   0.13   0.00   0.06   0.32   0.00   0.085
bcl      0.00   0.07   0.00   0.13   0.32   0.00   0.086
hil      0.00   0.18   0.00   0.09   0.32   0.00   0.098
blf      0.02   0.20   0.01   0.04   0.32   0.00   0.098
pam      0.02   0.27   0.00   0.06   0.25   0.00   0.098
duo      0.00   0.34   0.00   0.02   0.25   0.00   0.102
snv      0.00   0.44   0.01   0.17   0.00   0.00   0.105
pau      0.03   0.15   0.02   0.28   0.17   0.00   0.108
ilo      0.00   0.27   0.00   0.08   

In [23]:
results = list()
for tup in close_langs:
    
    results.append((tup[0], id2name[tup[0]], tup[1]))
results

[('tsg', 'tausug (suluk)', np.float64(0.03433096639872579)),
 ('cha', 'chamorro', np.float64(0.03925644198081491)),
 ('ivv', 'ivatan', np.float64(0.05555252261367141)),
 ('tiy', 'tiruray', np.float64(0.05612195697470507)),
 ('ceb', 'cebuano', np.float64(0.06017536459197514)),
 ('mrw', 'maranao', np.float64(0.07451979340324966)),
 ('akl', 'aklan', np.float64(0.08525016733874556)),
 ('bcl', 'bikol (bicolano)', np.float64(0.08583420334836683)),
 ('hil', 'hiligaynon', np.float64(0.09784054062144339)),
 ('blf', 'buol', np.float64(0.09810960572186206)),
 ('pam', 'kapampangan', np.float64(0.09821598500793216)),
 ('duo',
  'dupaningan agta; eastern cagayan agta; dupaninan agta',
  np.float64(0.10194836355327619)),
 ('snv', "sa'ban", np.float64(0.10478914190142201)),
 ('pau', 'palauan', np.float64(0.10798578005800759)),
 ('ilo', 'ilocano', np.float64(0.11535111591351697)),
 ('mad', 'madurese', np.float64(0.1168441735422502)),
 ('btx', 'karo batak', np.float64(0.11690492549431825)),
 ('bdl', 'sa